In [17]:
#!/usr/bin/env python3
"""
SyntOn Hit-Derived Library Enumeration Pipeline - FIXED VERSION
=============================================================
Imports building blocks from CSV, generates synthons using Synt-On's
mainSynthonsGenerator, and enumerates using the proper Synt-On API.
"""

import os
import sys
import logging
import time
import pandas as pd
from typing import List, Dict, Set, Tuple, Optional
from rdkit import Chem, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize as MolStd
from rdkit.Chem import Descriptors

RDLogger.DisableLog("rdApp.*")

# ── Synt-On Import ──
SYNTON_ROOT = os.path.join(os.getcwd(), "Synt-On")
if not os.path.exists(SYNTON_ROOT):
    SYNTON_ROOT = os.path.join(os.path.dirname(os.getcwd()), "Synt-On")
sys.path.insert(0, SYNTON_ROOT)

try:
    from src.SyntOn_BBs import mainSynthonsGenerator
    from src.SyntOn import enumeration, fragmentation
    print(f"✓ Successfully imported Synt-On modules from {SYNTON_ROOT}")
except ImportError as e:
    print(f"Could not import Synt-On modules: {e}")
    sys.exit(1)

# ── Configuration ──
HITS_THRESHOLD  = 3
SCORE_THRESHOLD = 0.460
CHUNK_SIZE      = 5000
MW_UPPER        = 1000
MW_LOWER        = 100
N_CORES         = 4
MAX_NEW_MOLS    = 100000  # Max molecules to generate

INPUT_CSV    = "Building_blocks_screened.csv"
OUTPUT_CSV   = "hit_recombination_library.csv"
SYNTHON_DB   = "synthon_database.csv"
OUT_DIR      = os.path.abspath("synton_library")

# ── Logging Setup ──
def setup_logging() -> logging.Logger:
    logger = logging.getLogger("SyntOn_Pipeline")
    logger.setLevel(logging.INFO)
    if not logger.handlers:
        ch = logging.StreamHandler(sys.stdout)
        ch.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", "%H:%M:%S"))
        logger.addHandler(ch)
    return logger

logger = setup_logging()

# ── Helper Functions ──
def standardize_smiles(smiles: str) -> Optional[str]:
    """Sanitize, keep largest fragment, neutralize charges, remove stereochemistry for consistency."""
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        mol = MolStd.LargestFragmentChooser().choose(mol)
        mol = MolStd.Uncharger().uncharge(mol)
        # Remove stereochemistry to avoid issues with labeled synthons
        Chem.RemoveStereochemistry(mol)
        return Chem.MolToSmiles(mol, isomericSmiles=False)
    except:
        return None

def has_reactive_label(smiles: str) -> bool:
    """Check if SMILES contains reactive labels (:NN format used by Synt-On)."""
    import re
    return bool(re.search(r':\d+', smiles))

# ── Step 1: Load & Filter Building Blocks ──
def load_and_filter(csv_path: str) -> pd.DataFrame:
    """Load building blocks CSV and filter by hit criteria."""
    logger.info("=" * 70)
    logger.info("STEP 1: LOADING & FILTERING BUILDING BLOCKS")
    logger.info("=" * 70)

    if not os.path.exists(csv_path):
        logger.error(f"File not found: {csv_path}")
        return pd.DataFrame()

    df = pd.read_csv(csv_path)
    logger.info(f"Total rows loaded : {len(df):,}")
    logger.info(f"Columns found     : {list(df.columns)}")

    # Check for required columns - SMILES is in 3rd column (index 2)
    # n_hits in 5th (index 4), fit_score in 4th (index 3)
    cols = df.columns.tolist()
    
    if len(cols) < 5:
        logger.error(f"Not enough columns. Expected at least 5, got {len(cols)}")
        return pd.DataFrame()
    
    # Rename columns for consistency
    smiles_col = cols[2]   # Third column
    fit_col = cols[3]      # Fourth column
    hits_col = cols[4]     # Fifth column
    
    df = df.rename(columns={
        smiles_col: 'smiles',
        fit_col: 'fit_score',
        hits_col: 'n_hits'
    })
    
    logger.info(f"Using '{smiles_col}' as SMILES column")
    logger.info(f"Using '{fit_col}' as fit_score column")
    logger.info(f"Using '{hits_col}' as n_hits column")
    
    # Drop rows with invalid SMILES
    df = df.dropna(subset=['smiles'])
    df = df[df['smiles'].apply(lambda x: isinstance(x, str) and len(x) > 0)]
    
    # Apply filters
    df['n_hits'] = pd.to_numeric(df['n_hits'], errors='coerce')
    df['fit_score'] = pd.to_numeric(df['fit_score'], errors='coerce')
    
    mask = (df["n_hits"] >= HITS_THRESHOLD) & (df["fit_score"] >= SCORE_THRESHOLD)
    df_hits = df[mask].copy().reset_index(drop=True)
    
    logger.info(f"After filters     : {len(df_hits):,} (n_hits>={HITS_THRESHOLD}, fit_score>={SCORE_THRESHOLD})")
    
    return df_hits

# ── Step 2: Generate Synthons Using Synt-On ──
def generate_synthons_from_building_blocks(df_blocks: pd.DataFrame) -> Tuple[List[str], pd.DataFrame]:
    """
    Generate labeled synthons from building blocks using Synt-On's mainSynthonsGenerator.
    This properly identifies reactive sites and adds reaction labels.
    """
    logger.info("=" * 70)
    logger.info(f"STEP 2: GENERATING SYNTHONS FROM {len(df_blocks):,} BUILDING BLOCKS")
    logger.info("=" * 70)

    all_synthons = []
    synthon_records = []
    failed_count = 0
    no_synthon_count = 0
    
    for idx, row in df_blocks.iterrows():
        if (idx + 1) % 50 == 0 or idx == 0:
            logger.info(f"  Processing building block {idx + 1}/{len(df_blocks)}...")
        
        smiles = str(row['smiles'])
        clean_smi = standardize_smiles(smiles)
        
        if clean_smi is None:
            failed_count += 1
            continue
        
        try:
            # Use Synt-On's mainSynthonsGenerator to identify and label reactive sites
            # returnDict=True gives us a dict of synthon SMILES -> building block info
            synthon_dict = mainSynthonsGenerator(
                initSmi=clean_smi,
                keepPG=False,
                Classes=None,
                returnDict=True,
                returnBoolAndDict=False
            )
            
            if not synthon_dict:
                no_synthon_count += 1
                logger.debug(f"No reactive sites found for {clean_smi[:60]}...")
                continue
            
            # Store unique labeled synthons
            for synthon_smi, bb_info in synthon_dict.items():
                if synthon_smi not in all_synthons:
                    all_synthons.append(synthon_smi)
                    
                    # Determine reactivity based on labels
                    # Synt-On typically uses :1, :2, :3, :4 for different reaction centers
                    # or :10, :20, :30, :40 depending on version
                    reactivity = "unknown"
                    if has_reactive_label(synthon_smi):
                        # Count types of labels
                        labels = set()
                        import re
                        found_labels = re.findall(r':(\d+)', synthon_smi)
                        labels = set(int(l) for l in found_labels)
                        
                        # Classify based on label numbers
                        has_electrophile = any(l in [1, 10, 3, 30] for l in labels)
                        has_nucleophile = any(l in [2, 20, 4, 40] for l in labels)
                        
                        if has_electrophile and has_nucleophile:
                            reactivity = "bifunctional"
                        elif has_electrophile:
                            reactivity = "electrophile"
                        elif has_nucleophile:
                            reactivity = "nucleophile"
                    
                    synthon_records.append({
                        'synthon_smiles': synthon_smi,
                        'original_smiles': clean_smi,
                        'parent_fit_score': row.get('fit_score', 0),
                        'parent_n_hits': row.get('n_hits', 0),
                        'reactivity': reactivity,
                        'bb_info': str(bb_info)
                    })
                    
        except Exception as e:
            logger.debug(f"Error processing building block {idx} ({clean_smi[:50]}...): {e}")
            failed_count += 1
    
    synthon_df = pd.DataFrame(synthon_records)
    
    if synthon_df.empty:
        logger.error("No synthons generated from any building block!")
        logger.info("This might mean:")
        logger.info("  1. Building blocks have no reactive sites recognizable by Synt-On")
        logger.info("  2. The molecules need different reaction rules")
        logger.info("  3. Check the SMILES formatting")
        return [], synthon_df
    
    # Save synthon database for inspection
    synthon_df.to_csv(SYNTHON_DB, index=False)
    
    logger.info(f"\nSynthon generation complete:")
    logger.info(f"  Building blocks processed : {len(df_blocks):,}")
    logger.info(f"  Unique synthons generated : {len(all_synthons):,}")
    logger.info(f"  Failed to process         : {failed_count}")
    logger.info(f"  No reactive sites found   : {no_synthon_count}")
    
    if not synthon_df.empty:
        logger.info(f"\nReactivity distribution:")
        for rxn_type, count in synthon_df['reactivity'].value_counts().items():
            logger.info(f"  {rxn_type:15} : {count}")
    
    logger.info(f"\n✓ Synthon database saved → '{SYNTHON_DB}'")
    
    return all_synthons, synthon_df

# ── Step 3: Get Reaction SMARTS from Synt-On ──
def get_reaction_smarts(reactions: str = "R1-R13") -> List[str]:
    """
    Initialize Synt-On's fragmentation to get the reaction SMARTS
    used for reconstruction/enumeration.
    """
    logger.info("=" * 70)
    logger.info("STEP 3: GETTING REACTION SMARTS FROM SYNT-ON")
    logger.info("=" * 70)
    
    try:
        logger.info(f"Initializing fragmentation with reactions: {reactions}")
        
        # Use the correct fragmentation signature based on earlier inspection:
        # fragmentation(fragmentationMode='use_all', reactionsToWorkWith='R1-R13', 
        #               maxNumberOfReactionCentersPerFragment=3, MaxNumberOfStages=5, ...)
        frag = fragmentation(
            fragmentationMode='use_all',
            reactionsToWorkWith=reactions,
            maxNumberOfReactionCentersPerFragment=3,
            MaxNumberOfStages=5
        )
        
        # Get reaction SMARTS for reconstruction (method confirmed earlier)
        reaction_smarts = frag.getReactionForReconstruction()
        
        logger.info(f"✓ Got {len(reaction_smarts)} reaction SMARTS from Synt-On")
        
        # Display first few for verification
        for i, rxn in enumerate(reaction_smarts[:3]):
            logger.info(f"  Reaction {i+1}: {rxn[:100]}...")
        
        return reaction_smarts
        
    except Exception as e:
        logger.error(f"Failed to get reaction SMARTS: {e}")
        logger.warning("Using default amide bond formation SMARTS as fallback")
        import traceback
        traceback.print_exc()
        
        # Fallback to common reaction SMARTS
        fallback_smarts = [
            "[C:1](=[O:2])-[OH:3].[N:4]>>[C:1](=[O:2])-[N:4]",
            "[C:1](=[O:2])-[Cl:3].[N:4]>>[C:1](=[O:2])-[N:4]",
        ]
        logger.info(f"Using {len(fallback_smarts)} fallback reaction SMARTS")
        return fallback_smarts

# ── Step 4: Enumerate Using Synt-On's Enumeration Engine ──
def enumerate_with_synton(synthons: List[str], reaction_smarts: List[str]) -> List[str]:
    """
    Use Synt-On's enumeration engine to generate products from labeled synthons.
    This simulates de novo reactions between compatible building blocks.
    """
    logger.info("=" * 70)
    logger.info(f"STEP 4: ENUMERATING WITH SYNT-ON ENGINE")
    logger.info("=" * 70)
    logger.info(f"  Labeled synthons : {len(synthons):,}")
    logger.info(f"  Reaction SMARTS  : {len(reaction_smarts)}")
    logger.info(f"  MW range         : {MW_LOWER} – {MW_UPPER}")
    logger.info(f"  Max new mols     : {MAX_NEW_MOLS:,}")
    logger.info(f"  CPU cores        : {N_CORES}")
    logger.info(f"  Output directory : {OUT_DIR}")
    
    os.makedirs(OUT_DIR, exist_ok=True)
    
    try:
        logger.info("Initializing enumeration engine...")
        
        # Use the confirmed enumeration signature:
        # enumeration(outDir, Synthons=None, reactionSMARTS=None, maxNumberOfReactedSynthons=6, 
        #             MWupperTh=None, MWlowerTh=None, desiredNumberOfNewMols=1000, nCores=1, 
        #             analoguesEnumeration=False)
        enumerator = enumeration(
            outDir=OUT_DIR,
            Synthons=synthons,
            reactionSMARTS=reaction_smarts,
            maxNumberOfReactedSynthons=6,
            MWupperTh=MW_UPPER,
            MWlowerTh=MW_LOWER,
            desiredNumberOfNewMols=MAX_NEW_MOLS,
            nCores=N_CORES,
            analoguesEnumeration=False  # De novo enumeration, not analogue generation
        )
        
        logger.info("Running enumeration engine (this may take several minutes)...")
        start_time = time.time()
        
        # Get reconstructed molecules using the confirmed method
        results = enumerator.getReconstructedMols()
        
        elapsed = time.time() - start_time
        logger.info(f"✓ Enumeration completed in {elapsed:.2f} seconds")
        
        if results:
            logger.info(f"  Generated {len(results):,} raw results")
        else:
            logger.warning("No molecules were generated by enumeration")
        
        return results if results else []
        
    except Exception as e:
        logger.error(f"Enumeration failed: {e}")
        import traceback
        traceback.print_exc()
        return []

def process_enumeration_results(results: List, original_synthons: List[str]) -> List[str]:
    """
    Process enumeration results into clean, unique SMILES strings.
    Handles RDKit Mol objects and string representations.
    """
    logger.info("=" * 70)
    logger.info("STEP 5: PROCESSING ENUMERATION RESULTS")
    logger.info("=" * 70)
    
    unique_smiles = set()
    mol_objects = 0
    string_objects = 0
    failed = 0
    
    for item in results:
        try:
            if isinstance(item, Chem.rdchem.Mol):
                mol_objects += 1
                # Clean the molecule
                try:
                    mol = MolStd.LargestFragmentChooser().choose(item)
                    mol = MolStd.Uncharger().uncharge(mol)
                    Chem.RemoveStereochemistry(mol)
                    smiles = Chem.MolToSmiles(mol, isomericSmiles=False)
                except:
                    smiles = Chem.MolToSmiles(item)
                
                if smiles:
                    unique_smiles.add(smiles)
            elif isinstance(item, str):
                string_objects += 1
                # Try to parse as SMILES
                mol = Chem.MolFromSmiles(item)
                if mol:
                    try:
                        mol = MolStd.LargestFragmentChooser().choose(mol)
                        mol = MolStd.Uncharger().uncharge(mol)
                        Chem.RemoveStereochemistry(mol)
                        smiles = Chem.MolToSmiles(mol, isomericSmiles=False)
                    except:
                        smiles = item
                    unique_smiles.add(smiles)
                else:
                    failed += 1
            else:
                failed += 1
        except Exception as e:
            logger.debug(f"Could not process result item: {e}")
            failed += 1
    
    logger.info(f"  RDKit Mol objects processed : {mol_objects:,}")
    logger.info(f"  String objects processed    : {string_objects:,}")
    logger.info(f"  Failed conversions          : {failed:,}")
    logger.info(f"  Unique valid SMILES         : {len(unique_smiles):,}")
    
    # If enumeration produced nothing useful, fall back to original synthons
    if not unique_smiles:
        logger.warning("No valid molecules from enumeration! Returning original synthons.")
        for s in original_synthons:
            cleaned = standardize_smiles(s)
            if cleaned:
                unique_smiles.add(cleaned)
    
    # Apply MW filters to final set
    filtered_smiles = set()
    for smi in unique_smiles:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mw = Descriptors.MolWt(mol)
            if MW_LOWER <= mw <= MW_UPPER:
                filtered_smiles.add(smi)
    
    logger.info(f"  After MW filter ({MW_LOWER}-{MW_UPPER}): {len(filtered_smiles):,}")
    
    return list(filtered_smiles)

# ── Step 6: Save Final Library ──
def save_library(smiles_list: List[str], output_csv: str):
    """
    Save the enumerated library to CSV with compound information.
    """
    logger.info("=" * 70)
    logger.info("STEP 6: SAVING FINAL LIBRARY")
    logger.info("=" * 70)
    logger.info(f"Total compounds to save: {len(smiles_list):,}")
    
    total = len(smiles_list)
    
    for chunk_start in range(0, total, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, total)
        chunk = smiles_list[chunk_start:chunk_end]
        
        # Calculate molecular weights
        mw_list = []
        for smi in chunk:
            mol = Chem.MolFromSmiles(smi)
            mw = Descriptors.MolWt(mol) if mol else 0
            mw_list.append(mw)
        
        # Create dataframe
        df = pd.DataFrame({
            "smiles": chunk,
            "compound_id": [f"HIT_ENUM_{chunk_start + i:08d}" for i in range(len(chunk))],
            "source": "hit_derived_enumeration",
            "mw": mw_list
        })
        
        # Write to CSV
        mode = 'w' if chunk_start == 0 else 'a'
        header = (chunk_start == 0)
        df.to_csv(output_csv, mode=mode, index=False, header=header)
        
        if chunk_start % (CHUNK_SIZE * 5) == 0:
            logger.info(f"  Saved {chunk_end:,}/{total:,} compounds")
    
    # Calculate statistics
    mw_values = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            mw_values.append(Descriptors.MolWt(mol))
    
    # Generate summary file
    stats_file = output_csv.replace('.csv', '_summary.txt')
    with open(stats_file, 'w') as f:
        f.write(f"Synt-On Hit-Derived Library Enumeration Summary\n")
        f.write(f"{'='*55}\n")
        f.write(f"Total compounds generated : {total:,}\n")
        f.write(f"Output file               : {output_csv}\n")
        f.write(f"Date generated            : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
        if mw_values:
            f.write(f"\nMolecular Weight Statistics:\n")
            f.write(f"  Min MW  : {min(mw_values):.2f}\n")
            f.write(f"  Max MW  : {max(mw_values):.2f}\n")
            f.write(f"  Mean MW : {sum(mw_values)/len(mw_values):.2f}\n")
            f.write(f"  Median MW : {sorted(mw_values)[len(mw_values)//2]:.2f}\n")
    
    logger.info(f"\n✓ Library saved to '{output_csv}'")
    logger.info(f"✓ Summary saved to '{stats_file}'")

# ── Main Pipeline ──
def main():
    logger.info("=" * 70)
    logger.info("SYNT-ON HIT-DERIVED LIBRARY ENUMERATION PIPELINE")
    logger.info("USING SYNT-ON'S NATIVE ENUMERATION ENGINE")
    logger.info("=" * 70)
    logger.info(f"Configuration:")
    logger.info(f"  Input CSV          : {INPUT_CSV}")
    logger.info(f"  Output CSV         : {OUTPUT_CSV}")
    logger.info(f"  Synthon DB         : {SYNTHON_DB}")
    logger.info(f"  Output directory   : {OUT_DIR}")
    logger.info(f"  Hit threshold      : n_hits >= {HITS_THRESHOLD}")
    logger.info(f"  Score threshold    : fit_score >= {SCORE_THRESHOLD}")
    logger.info(f"  MW range           : {MW_LOWER} - {MW_UPPER}")
    logger.info(f"  CPU cores          : {N_CORES}")
    logger.info(f"  Max new molecules  : {MAX_NEW_MOLS:,}")
    logger.info("=" * 70)
    
    # Step 1: Load and filter building blocks
    df_blocks = load_and_filter(INPUT_CSV)
    if df_blocks.empty:
        logger.error("No building blocks passed filters. Exiting.")
        logger.info("Check your thresholds or input file.")
        return
    
    logger.info(f"\n✓ Loaded {len(df_blocks):,} building blocks after filtering")
    
    # Step 2: Generate labeled synthons from building blocks
    synthons, synthon_df = generate_synthons_from_building_blocks(df_blocks)
    if not synthons:
        logger.error("No synthons generated. Cannot proceed with enumeration.")
        logger.info("Possible issues:")
        logger.info("  - Building blocks may not contain recognizable reactive groups")
        logger.info("  - Try different reaction rules or check SMILES format")
        return
    
    logger.info(f"\n✓ Generated {len(synthons):,} labeled synthons")
    
    # Step 3: Get reaction SMARTS from Synt-On fragmentation
    reaction_smarts = get_reaction_smarts("R1-R13")
    if not reaction_smarts:
        logger.error("No reaction SMARTS available. Cannot enumerate.")
        return
    
    # Step 4: Enumerate products using Synt-On's engine
    enumeration_results = enumerate_with_synton(synthons, reaction_smarts)
    
    # Step 5: Process and clean results
    final_smiles = process_enumeration_results(enumeration_results, synthons)
    
    if not final_smiles:
        logger.error("No valid molecules produced after processing.")
        return
    
    # Step 6: Save the final library
    save_library(final_smiles, OUTPUT_CSV)
    
    # Final summary
    logger.info("\n" + "=" * 70)
    logger.info("PIPELINE COMPLETED SUCCESSFULLY")
    logger.info("=" * 70)
    logger.info(f"  Building blocks input   : {len(df_blocks):,}")
    logger.info(f"  Labeled synthons        : {len(synthons):,}")
    logger.info(f"  Reaction SMARTS used    : {len(reaction_smarts)}")
    logger.info(f"  Final compounds         : {len(final_smiles):,}")
    logger.info(f"\nOutput files:")
    logger.info(f"  - {SYNTHON_DB} (intermediate synthons)")
    logger.info(f"  - {OUTPUT_CSV} (final enumerated library)")
    logger.info(f"  - {OUTPUT_CSV.replace('.csv', '_summary.txt')} (statistics)")
    logger.info(f"  - {OUT_DIR}/ (Synt-On intermediate files)")
    logger.info("=" * 70)

if __name__ == "__main__":
    main()

✓ Successfully imported Synt-On modules from /home/z.budagov/Desktop/cytochrome/CytochromeOxidasebd1ZB/Synt-On
[15:24:16] ======================================================================


INFO:SyntOn_Pipeline:======================================================================


[15:24:16] SYNT-ON HIT-DERIVED LIBRARY ENUMERATION PIPELINE


INFO:SyntOn_Pipeline:SYNT-ON HIT-DERIVED LIBRARY ENUMERATION PIPELINE


[15:24:16] USING SYNT-ON'S NATIVE ENUMERATION ENGINE


...

INFO:SyntOn_Pipeline:  - hit_recombination_library_summary.txt (statistics)


[16:07:19]   - /home/z.budagov/Desktop/cytochrome/CytochromeOxidasebd1ZB/synton_library/ (Synt-On intermediate files)


INFO:SyntOn_Pipeline:  - /home/z.budagov/Desktop/cytochrome/CytochromeOxidasebd1ZB/synton_library/ (Synt-On intermediate files)


[16:07:19] ======================================================================


INFO:SyntOn_Pipeline:======================================================================
